# Z3-Rust 09 — L'enigme d'Einstein (Zebra puzzle)

**Navigation** : [Index Z3-API](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Serie Z3 C#](../Z3-Linq2Z3/README.md) | [<< 08 - Ordonnancement](Z3-08-Ordonnancement-Python.ipynb) | [→ 10 - Cryptarithmes](Z3-10-Cryptarithmetic-Python.ipynb) | [Jumeau Python](Z3-09-Enigme-Einstein-Python.ipynb) | [Jumeau C#](Z3-09-Enigme-Einstein-CSharp.ipynb)

> **Solveur** : Z3 — crate Rust `z3` 0.24.0, exécuté dans le noyau `rust` (evcxr). Le portage suit le **même fil pédagogique** que le jumeau Python : encodage par position → résolution et témoin → benchmark et unicité → trois exemples guidés → trois exercices.

## Le problème

L'**enigme d'Einstein** (ou *Zebra puzzle*) est un classique de logique : 5 maisons alignees, chacune habitee par une personne de nationalite, couleur, boisson, cigarette et animal différents. On dispose de **15 indices** (le Britannique vit dans la rouge, le Danois boit du the, la verte est a gauche de la blanche, etc.) et la question est : **qui possede le poisson ?**

A la main, la resolution procede par deductions progressives sur un tableau - c'est long et trompeur. L'espace de recherche brut est enorme : chaque attribut est une permutation des 5 maisons, soit $(5!)^5 = 24\,883\,200\,000$ combinaisons. Une **enumeration exhaustive** (brute force) testant les 15 indices pour chaque combinaison prend plusieurs minutes.

C'est typiquement la ou un solveur SMT comme Z3 **fait la différence** : on decrit les 25 variables (position de chaque valeur), le domaine, les 5 contraintes de permutation (`Distinct`) et les 15 indices, et `Solver` trouve l'unique solution en **quelques millisecondes**. Ce notebook porte en Rust l'exemple C# [18_Einsteins_Riddle](../Z3-Linq2Z3/18_Einsteins_Riddle.ipynb) de la serie sœur Z3.Linq.

> **Note de portage** : la crate `z3` expose la même API SMT que le paquet Python `z3-solver`, mais l'égalité Rust n'est pas l'égalité SMT (`x == 10` compare deux valeurs Rust, la contrainte s'écrit `Ast::eq(&x, &Int::from_i64(10))`), et le contexte est **implicitement thread-local** depuis la version 0.19 (`Solver::new()`, `Int::new_const("x")` — plus de `Context` explicite).


In [2]:
:dep z3 = "0.24.0"

// Imports et vérification de l'environnement.
//
// Le noyau `rust` (evcxr) compile chaque cellule : les `fn` persistent d'une
// cellule a l'autre, les variables locales (`let`) et les `use` non. D'ou le
// patron du carnet : chaque cellule de calcul redonne ses propres `use` et
// reconstruit son etat local a partir des items persistants.

// La version du moteur est relevee a l'execution : `z3::full_version()` est
// reexporte a la racine du crate (le module `version` n'est pas un point
// d'entree public).
println!("Imports OK : z3 crate 0.24.0 (libz3 {})", z3::full_version());


Imports OK : z3 crate 0.24.0 (libz3 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e)


## Modelisation : l'encodage par position

L'astuce centrale consiste a **inverser le point de vue**. Au lieu de demander « quelle couleur a la maison *k* ? », on définit pour chaque valeur d'attribut une variable entiere egale a la **position** (numéro de maison 0..4) ou elle se trouve :

- `Brit`, `Swede`, `Dane`, `Norwegian`, `German` : position de chaque nationalite
- `Red`, `Green`, `White`, `Yellow`, `Blue` : position de chaque couleur
- `Tea`, `Coffee`, `Milk`, `Water`, `Beer` : position de chaque boisson
- `PallMall`, `Dunhill`, `Blend`, `BlueMaster`, `Prince` : position de chaque cigarette
- `Dog`, `Bird`, `Cat`, `Horse`, `Fish` : position de chaque animal

Soit **25 variables entieres**. Les contraintes se traduisent alors très naturellement :

1. **Domaine** : chaque variable $\in \{0,1,2,3,4\}$.
2. **Permutation** : dans chaque groupe d'attributs, les 5 positions sont deux a deux distinctes (`Distinct`) - chaque valeur occupe une maison différente.
3. **Les 15 indices** : des egalites de position (`Brit == Red` : le Britannique et la rouge sont dans la même maison) ou des **adjacences** (`|Blend - Cat| == 1` : le fumeur de Blend est voisin du Chat).

In [3]:
// Construit le solveur Z3 de l'enigme d'Einstein (25 variables de position,
// 15 indices). Retourne (solveur, variables, groupes) pour reuse dans les exercices.
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

fn build_einstein_solver() -> (Solver, HashMap<String, Int>, Vec<(&'static str, Vec<&'static str>)>) {
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let groups: Vec<(&str, Vec<&str>)> = vec![
        ("Nationalite", vec!["Brit", "Swede", "Dane", "Norwegian", "German"]),
        ("Couleur", vec!["Red", "Green", "White", "Yellow", "Blue"]),
        ("Boisson", vec!["Tea", "Coffee", "Milk", "Water", "Beer"]),
        ("Cigarette", vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"]),
        ("Animal", vec!["Dog", "Bird", "Cat", "Horse", "Fish"]),
    ];

    let mut v: HashMap<String, Int> = HashMap::new();
    for (_groupe, valeurs) in groups.iter() {
        for nom in valeurs {
            v.insert(nom.to_string(), Int::new_const(*nom));
        }
    }

    let mut s = Solver::new();

    // (1) Domaine 0..4 pour les 25 variables
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }

    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes
    for (_groupe, valeurs) in groups.iter() {
        let vars: Vec<Int> = valeurs.iter().map(|nom| v[*nom].clone()).collect();
        s.assert(&Ast::distinct(&vars));
    }

    // (3) Les 15 indices
    s.assert(&Ast::eq(&v["Brit"], &v["Red"]));            // 1. le Britannique vit dans la rouge
    s.assert(&Ast::eq(&v["Swede"], &v["Dog"]));           // 2. le Suedois eleve des chiens
    s.assert(&Ast::eq(&v["Dane"], &v["Tea"]));            // 3. le Danois boit du the
    s.assert(&Ast::eq(&v["Green"], &Int::sub(&[&v["White"], &Int::from_i64(1)])));   // 4. la verte est juste a gauche de la blanche
    s.assert(&Ast::eq(&v["Green"], &v["Coffee"]));        // 5. la verte boit le cafe
    s.assert(&Ast::eq(&v["PallMall"], &v["Bird"]));       // 6. Pall Mall eleve des oiseaux
    s.assert(&Ast::eq(&v["Yellow"], &v["Dunhill"]));      // 7. la jaune fume du Dunhill
    s.assert(&Ast::eq(&v["Milk"], &Int::from_i64(2)));    // 8. la maison du milieu boit du lait
    s.assert(&Ast::eq(&v["Norwegian"], &Int::from_i64(0)));   // 9. le Norvegien dans la 1ere maison
    s.assert(&Bool::or(&[                                     // 10. Blend voisin du Chat
        &Ast::eq(&Int::sub(&[&v["Blend"], &v["Cat"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Cat"], &v["Blend"]]), &Int::from_i64(1)),
    ]));
    s.assert(&Bool::or(&[                                     // 11. Cheval voisin du Dunhill
        &Ast::eq(&Int::sub(&[&v["Horse"], &v["Dunhill"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Dunhill"], &v["Horse"]]), &Int::from_i64(1)),
    ]));
    s.assert(&Ast::eq(&v["BlueMaster"], &v["Beer"]));     // 12. BlueMaster boit de la biere
    s.assert(&Ast::eq(&v["German"], &v["Prince"]));       // 13. l'Allemand fume du Prince
    s.assert(&Bool::or(&[                                     // 14. Norvegien voisin de la bleue
        &Ast::eq(&Int::sub(&[&v["Norwegian"], &v["Blue"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Blue"], &v["Norwegian"]]), &Int::from_i64(1)),
    ]));
    s.assert(&Bool::or(&[                                     // 15. Blend voisin de l'Eau
        &Ast::eq(&Int::sub(&[&v["Blend"], &v["Water"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Water"], &v["Blend"]]), &Int::from_i64(1)),
    ]));

    (s, v, groups)
}

let (s, v, groups) = build_einstein_solver();
println!("Modele encode : {} variables de position, {} groupes Distinct, 15 indices.",
         v.len(), groups.len());


Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.


## Resolution et temoin



On appelle `s.check()` : si `sat`, on extrait le modèle puis on **inverse** l'encodage par position - pour chaque maison $h \in \{0..4\}$, on cherche quelle valeur de chaque attribut y se trouve (la valeur dont la position vaut $h$). Cela reconstitue le tableau classique de la solution.

In [4]:
// Labels francais pour l'affichage (associes au nom de variable).
use std::collections::HashMap;
use z3::ast::Int;
use z3::{SatResult, Solver};

// Les cinq groupes et leurs valeurs, dans l'ordre d'affichage.
fn labels_fr() -> Vec<(&'static str, Vec<(&'static str, &'static str)>)> {
    vec![
        ("Nationalite", vec![("Britannique", "Brit"), ("Suedois", "Swede"), ("Danois", "Dane"), ("Norvegien", "Norwegian"), ("Allemand", "German")]),
        ("Couleur", vec![("Rouge", "Red"), ("Verte", "Green"), ("Blanche", "White"), ("Jaune", "Yellow"), ("Bleue", "Blue")]),
        ("Boisson", vec![("The", "Tea"), ("Cafe", "Coffee"), ("Lait", "Milk"), ("Eau", "Water"), ("Biere", "Beer")]),
        ("Cigarette", vec![("Pall Mall", "PallMall"), ("Dunhill", "Dunhill"), ("Blend", "Blend"), ("BlueMaster", "BlueMaster"), ("Prince", "Prince")]),
        ("Animal", vec![("Chien", "Dog"), ("Oiseau", "Bird"), ("Chat", "Cat"), ("Cheval", "Horse"), ("Poisson", "Fish")]),
    ]
}

// Resout l'enigme et affiche le tableau des 5 maisons.
fn solve_and_display() -> Option<(z3::Model, HashMap<String, Int>)> {
    let (s, v, _) = build_einstein_solver();
    let satisfiable = s.check() == SatResult::Sat;
    if !satisfiable {
        println!("UNSAT (inattendu pour cette enigme)");
    }
    let m = match s.get_model() {
        Some(m) => m,
        None => return None,
    };

    // nom de variable -> position (0..4)
    let mut pos: HashMap<String, i64> = HashMap::new();
    for (nom, var) in v.iter() {
        // `eval` rend une Option : une variable absente du modele donne None,
        // jamais une panique.
        if let Some(p) = m.eval(var, true).and_then(|x| x.as_i64()) {
            pos.insert(nom.clone(), p);
        }
    }

    // Label francais de la valeur du groupe presente a la maison h.
    let val_at = |groupe: &str, h: i64| -> String {
        for (fr, key) in labels_fr().iter().find(|(g, _)| *g == groupe).unwrap().1.iter() {
            if pos.get(*key) == Some(&h) {
                return fr.to_string();
            }
        }
        "?".to_string()
    };

    let cols = ["Maison", "Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"];
    let widths = [7usize, 13, 9, 9, 13, 9];
    let ligne = |mots: &[&str]| -> String {
        mots.iter()
            .zip(widths.iter())
            .map(|(mot, w)| format!("{:<width$}", mot, width = w))
            .collect::<Vec<String>>()
            .join("  ")
    };

    println!("{}", ligne(&cols));
    println!("  {}", "-".repeat(widths.iter().sum::<usize>() + 2 * (cols.len() - 1)));
    for h in 0..5i64 {
        let mut row: Vec<String> = vec![(h + 1).to_string()];
        for groupe in ["Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"] {
            row.push(val_at(groupe, h));
        }
        let refs: Vec<&str> = row.iter().map(|s| s.as_str()).collect();
        println!("{}", ligne(&refs));
    }
    println!();

    let pos_fish = pos.get("Fish").copied().unwrap_or(0);
    println!(">>> Reponse : c'est le {} qui possede le Poisson (maison {}).",
             val_at("Nationalite", pos_fish), pos_fish + 1);

    Some((m, v))
}

// Le resultat du `match` est lie a `_` : evcxr affiche la valeur de la
// derniere expression de la cellule, et un `match` non lie afficherait `()`.
let _ = match solve_and_display() {
    Some((_model, _vars)) => {}
    None => println!("UNSAT (inattendu pour cette enigme)"),
};


Maison   Nationalite    Couleur    Boisson    Cigarette      Animal   


  ----------------------------------------------------------------------


1        Norvegien      Jaune      Eau        Dunhill        Chat     


2        Danois         Bleue      The        Blend          Cheval   


3        Britannique    Rouge      Lait       Pall Mall      Oiseau   


4        Allemand       Verte      Cafe       Prince         Poisson  


5        Suedois        Blanche    Biere      BlueMaster     Chien    


>>> Reponse : c'est le Allemand qui possede le Poisson (maison 4).


## Interpretation

Z3 trouve l'unique solution en quelques millisecondes. La resolution humaine procederait par deductions progressives - on place le Norvegien en 1 (`Norwegian == 0`), le lait en 3 (`Milk == 2`), puis on propage les egalites et adjacences - mais c'est précisément ce travail de **propagation de contraintes** que le solveur automatise, sans erreur ni oubli.

La reponse a la question « qui possede le poisson ? » tombe directement du modèle extrait. Notons qu'**aucun indice ne parle du poisson** : c'est par elimination (les 4 autres animaux sont localises par les indices) que sa position se deduit.

## Vérification du témoin : relire les indices

Un modèle extrait n'est pas un résultat à prendre sur parole — vérifions que le tableau ci-dessus satisfait bien quelques indices choisis dans l'énoncé original :

- **Le Norvégien vit à côté de la maison bleue** : Norvégien en maison 1, Bleue en maison 2 — voisins.
- **Le fumeur de Blend a un voisin qui garde le chat** : Blend en maison 2, Chat en maison 1 — voisins.
- **La maison verte est immédiatement à gauche de la blanche** : Verte en 4, Blanche en 5.
- **La maison du centre boit du lait** : Lait en maison 3.

Ce geste de re-lecture croisée est l'équivalent d'un test unitaire sur le solveur : il valide l'**encodage** (une contrainte mal traduite produirait un témoin qui échoue à la vérification), pas seulement la satisfiabilité.

## Approche naive vs solveur : mesurer l'ecart



Plutot que d'affirmer que le solveur est plus rapide, **mesurons** l'ecart. L'espace de recherche brut est $(5!)^5$ : chaque attribut est une permutation des 5 maisons, independamment. On chronometre aussi la resolution Z3, puis on **prouve l'unicite** de la solution en ajoutant la negation du temoin trouve et en re-resolvant (on attend `unsat`).

In [5]:
// (1) Espace de recherche brut, (2) chronometre de la resolution Z3,
// (3) unicite de la solution.
use std::collections::HashMap;
use std::time::Instant;
use z3::ast::{Ast, Bool, Int};
use z3::{SatResult, Solver};

// Formatage d'un entier avec les separateurs de milliers (24,883,200,000).
fn avec_separateurs(n: u128) -> String {
    let chiffres: Vec<char> = n.to_string().chars().collect();
    let mut sortie = String::new();
    for (i, c) in chiffres.iter().enumerate() {
        if i > 0 && (chiffres.len() - i) % 3 == 0 {
            sortie.push(',');
        }
        sortie.push(*c);
    }
    sortie
}

// (1) Espace de recherche brut : (5!)^5 = 24_883_200_000
let space: u128 = 24_883_200_000;
let brute_seconds = space as f64 / 1e8;   // a ~1e8 tests/s, sans tester les 15 indices

// (2) Chronometre la resolution Z3
let t0 = Instant::now();
let (s1, v1, _) = build_einstein_solver();
let _ = s1.check();
let m1 = s1.get_model();
let z3_ms = t0.elapsed().as_secs_f64() * 1000.0;

// (3) Unicite : on nie le temoin trouve (au moins une variable differe) -> on attend unsat
let (mut s2, v2, _) = build_einstein_solver();
let mut alternatives: Vec<Bool> = Vec::new();
for (nom, var) in v2.iter() {
    let valeur = match &m1 {
        Some(mm) => mm.eval(&v1[nom], true).and_then(|x| x.as_i64()).unwrap_or(0),
        None => 0,
    };
    alternatives.push(var.ne(&Int::from_i64(valeur)));
}
s2.assert(&Bool::or(&alternatives.iter().collect::<Vec<&Bool>>()));
let unique = s2.check() == SatResult::Unsat;

println!("Espace de recherche brut       : (5!)^5 = {} combinaisons", avec_separateurs(space));
println!("Brute force estime (~1e8/s)    : {:.1} s (hors test des 15 indices)", brute_seconds);
println!("Z3 solve temps                 : {:.1} ms", z3_ms);
// Rust rend les booleens en minuscules ("true") ; le jumeau Python rend
// "True" -- on impose la forme du jumeau de reference (sortie comparable).
println!("Solution unique (UNSAT si nie) : {}", if unique { "True" } else { "False" });
println!(" -> Z3 resout en millisecondes un espace de ~{:.0} milliards, et prouve l'unicite sans enumeration.",
         space as f64 / 1e9);


Espace de recherche brut       : (5!)^5 = 24,883,200,000 combinaisons


Brute force estime (~1e8/s)    : 248.8 s (hors test des 15 indices)


Z3 solve temps                 : 13.3 ms


Solution unique (UNSAT si nie) : True


 -> Z3 resout en millisecondes un espace de ~25 milliards, et prouve l'unicite sans enumeration.


## Lecture du benchmark

Sur cette enigme, l'ecart est **frappant** : Z3 resout les 25 variables entrelacees en quelques millisecondes, alors que la brute force doit affronter **~25 milliards** de combinaisons (soit plusieurs minutes même a cadence elevee, et bien plus avec le test des 15 indices a chaque candidat).

La **propagation** fait tout le travail : `Norwegian == 0` et `Milk == 2` sont des ancres imposees par les indices 9 et 8, puis les egalites de position et les adjacences reduisent les domaines restants jusqu'a ce qu'il ne reste qu'une seule combinaison coherente. C'est exactement la mecanique qu'un humain deployerait a la main sur le tableau - mais le solveur la realise sans erreur et la **prouve complete** (unicite via `unsat` sur la negation).

## Exercices

Les trois exercices initiaux de ce carnet ont été résolus par un rendu de TP : chacun devient un **exemple guidé** (code de l'étudiante conservé tel quel), suivi d'une lecture du résultat puis d'un **nouvel exercice** non résolu, de difficulté comparable et portant le même concept Z3. Vous y réutiliserez l'encodage par position et la fonction `build_einstein_solver`.

### Exemple guidé 1 -- Un 16e indice : le Chat boit de l'Eau

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : ajouter l'indice `Cat == Water` (le Chat et l'Eau sont dans la même maison) et observer si l'énigme reste satisfaisable (`sat`) ou devient contradictoire (`unsat`).

**Rappel** : on reprend `build_einstein_solver()`, on ajoute `s.assert(&Ast::eq(&v["Cat"], &v["Water"]))` avant `check()`, et on compare avec la solution trouvée plus haut — le Chat et l'Eau y sont-ils dans la même maison ?

In [6]:
// Exemple guidé 1 : ajout de l'indice Cat == Water et test de la satisfiabilite.
// Etape 1 : reprendre build_einstein_solver() -> s, v.
// Etape 2 : ajouter l'indice Cat == Water.
// Etape 3 : verifier la satisfiabilite -> sat ou unsat ?
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::{SatResult, Solver};

// `SatResult` n'implement pas `Display` dans la crate 0.23 : on rend les
// memes chaines que le jumeau Python ('sat' / 'unsat' / 'unknown').
fn verdict_str(v: SatResult) -> &'static str {
    match v {
        SatResult::Sat => "sat",
        SatResult::Unsat => "unsat",
        SatResult::Unknown => "unknown",
    }
}

fn tester_chat_eau() -> SatResult {
    // Etape 1 : On recupere le solveur deja configure avec les 15 indices
    let (s, v, _) = build_einstein_solver();

    // Etape 2 : On ajoute le 16e indice (le Chat et l'Eau dans la meme maison)
    s.assert(&Ast::eq(&v["Cat"], &v["Water"]));

    // Etape 3 : On verifie si le probleme reste possible et on renvoie le resultat
    s.check()
}

let verdict = tester_chat_eau();
println!("Verdict avec Cat == Water : {}", verdict_str(verdict));


Verdict avec Cat == Water : sat


### Lecture du résultat : un indice déjà vrai

Le verdict est `sat` : l'énigme reste satisfaisable. Ce n'est pas un hasard — dans l'unique solution affichée plus haut, le Chat et l'Eau sont **déjà** dans la même maison (maison 1, chez le Norvégien). Le 16e indice est donc déjà vrai : il ne crée aucune contradiction et n'apporte aucune information nouvelle. Un indice qui contredirait la solution (par exemple `Fish == Water`) rendrait au contraire l'énigme `unsat` — c'est exactement l'objet du nouvel exercice 1.

### Exercice 1 -- Prédire le verdict d'un 17e indice

**Objectif** : généraliser l'exemple guidé 1 en une fonction qui classe **n'importe quel** indice candidat, puis répondre à la question conceptuelle.

Écrire `classer_indice(indice)` qui prend un indice candidat (une fonction de `v` retournant une expression booléenne Z3, sur le patron du bonus de l'exemple guidé 2), reconstruit le solveur complet, ajoute l'indice et retourne le verdict `SatResult::Sat` ou `SatResult::Unsat`. Classer ensuite les quatre candidats :

1. `|v: &HashMap<String, Int>| Ast::eq(&v["Cat"], &v["Water"])` — le Chat et l'Eau dans la même maison ;
2. `|v: &HashMap<String, Int>| Ast::eq(&v["Fish"], &v["Water"])` — le Poisson et l'Eau dans la même maison ;
3. `|v: &HashMap<String, Int>| Ast::eq(&v["Horse"], &v["Blend"])` — le Cheval dans la maison du fumeur de Blend ;
4. `|v: &HashMap<String, Int>| Ast::eq(&v["Bird"], &v["Milk"])` — l'Oiseau dans la maison au lait.

**Question** : parmi les verdicts obtenus, peut-il exister un indice qui donne `sat` tout en **changeant** la solution du tableau ? Justifier à partir de l'unicité prouvée plus haut.

**Indices** : chaque appel doit repartir d'un solveur **frais** (`build_einstein_solver()`) — sinon les indices s'accumulent d'un appel à l'autre. Pour la question : que vaut l'ensemble des solutions avant l'ajout, et que lui fait un indice déjà vrai ?

In [7]:
// EXERCICE 1 : classer un 17e indice candidat (sat / unsat).
// TODO etudiant :
// 1. ecrire fn classer_indice(indice: &dyn Fn(&HashMap<String, Int>) -> Bool) -> Option<SatResult>
//    qui repart d'un solveur FRAIS a chaque appel (build_einstein_solver()),
//    ajoute indice(&v) et retourne le verdict (SatResult::Sat ou SatResult::Unsat)
// 2. classer les 4 indices candidats de l'enonce :
//    |v: &HashMap<String, Int>| Ast::eq(&v["Cat"], &v["Water"])
//    |v: &HashMap<String, Int>| Ast::eq(&v["Fish"], &v["Water"])
//    |v: &HashMap<String, Int>| Ast::eq(&v["Horse"], &v["Blend"])
//    |v: &HashMap<String, Int>| Ast::eq(&v["Bird"], &v["Milk"])
// 3. repondre a la question -- un indice sat peut-il changer la solution ?
println!("Exercice a completer : classer les 4 indices candidats.");


Exercice a completer : classer les 4 indices candidats.


### Exemple guidé 2 -- Énumérer toutes les solutions par blocage itératif

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : énumérer **toutes** les solutions de l'énigme par **blocage itératif** (on résout, on compte, on ajoute une contrainte interdisant le témoin trouvé, on re-résout) jusqu'à `unsat`. Le compte attendu est 1 (solution unique), ce qui confirme le benchmark ci-dessus. Le rendu généralise la fonction en `compter_solutions(s=None, v=None, limite=1000)` pour qu'elle s'applique à n'importe quel solveur — elle resservira dans l'exemple guidé 3 et le nouvel exercice 2.

In [8]:
// Exemple guidé 2 : énumérer toutes les solutions par blocage itératif.
// Etape 1 : construire le solveur complet.
// Etape 2 : tant que check() == sat : compter, prendre le modele, interdire ce modele.
// Etape 3 : afficher le compte (attendu : 1).
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::{SatResult, Solver};

// Retourne le nombre total de solutions (attendu : 1 pour l'enigme d'Einstein).
// On peut aussi lui passer un autre solveur (s, v), par exemple la mini-enigme
// de l'exemple guide 3.
fn compter_solutions(s: &mut Solver, v: &HashMap<String, Int>, limite: usize) -> usize {
    let mut compte = 0usize;
    while compte < limite && s.check() == SatResult::Sat {
        compte += 1;
        let m = match s.get_model() {                          // la solution trouvee
            Some(m) => m,
            None => break,
        };
        let mut interdits: Vec<Bool> = Vec::new();
        for (_nom, var) in v.iter() {
            let valeur = m.eval(var, true).and_then(|x| x.as_i64()).unwrap_or(0);
            interdits.push(var.ne(&Int::from_i64(valeur)));
        }
        // on l'interdit : au moins 1 variable doit changer
        s.assert(&Bool::or(&interdits.iter().collect::<Vec<&Bool>>()));
    }
    compte
}

// Etape 3 : afficher le compte
let (mut s, v, _) = build_einstein_solver();
let nb = compter_solutions(&mut s, &v, 1000);
println!("Nombre de solutions de l'enigme d'Einstein : {} (attendu : 1)", nb);


Nombre de solutions de l'enigme d'Einstein : 1 (attendu : 1)


In [9]:
// Bonus exemple guidé 2 : la boucle compte-t-elle vraiment ? On retire un indice et on recompte.
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::{SatResult, Solver};

// Recree le solveur d'Einstein en retirant UNE contrainte (l'indice donne).
fn solveur_sans_indice(indice: &dyn Fn(&HashMap<String, Int>) -> Bool) -> (Solver, HashMap<String, Int>) {
    let (s, v, _) = build_einstein_solver();
    let s2 = Solver::new();
    let cible = indice(&v);
    for a in s.get_assertions() {
        // Comparaison STRUCTURELLE des assertions (`ast_eq`), et non l'egalite
        // SMT : on retire celle-la, on recopie toutes les autres.
        if !a.ast_eq(&cible) {
            s2.assert(&a);
        }
    }
    (s2, v)
}

let tests: Vec<(&str, Box<dyn Fn(&HashMap<String, Int>) -> Bool>)> = vec![
    ("9  (Norvegien en maison 1)", Box::new(|v: &HashMap<String, Int>| {
        Ast::eq(&v["Norwegian"], &Int::from_i64(0))
    })),
    ("15 (Blend voisin de l'Eau)", Box::new(|v: &HashMap<String, Int>| {
        Bool::or(&[
            &Ast::eq(&Int::sub(&[&v["Blend"], &v["Water"]]), &Int::from_i64(1)),
            &Ast::eq(&Int::sub(&[&v["Water"], &v["Blend"]]), &Int::from_i64(1)),
        ])
    })),
];

// La boucle est liee a `_` : evcxr affiche la valeur de la derniere
// expression de la cellule, et une boucle non liee afficherait `()`.
let _ = for (nom, indice) in tests.iter() {
    let (mut s2, v2) = solveur_sans_indice(indice.as_ref());
    let n = compter_solutions(&mut s2, &v2, 1000);
    println!("Sans l'indice {} : {} solution(s)", nom, n);
};


Sans l'indice 9  (Norvegien en maison 1) : 17 solution(s)


Sans l'indice 15 (Blend voisin de l'Eau) : 1 solution(s)


### Lecture du résultat : la boucle compte vraiment, et un indice est redondant

Le blocage itératif (trouver une solution, l'interdire par un `Or` sur les 25 variables, recommencer jusqu'à `unsat`) donne **1 solution** : l'énigme d'Einstein est bien unique, ce qui confirme le benchmark. Pour vérifier que la boucle compte réellement, on retire un indice et on recompte : sans l'indice 9 (Norvégien en maison 1), on trouve **17 solutions**. Sans l'indice 15 (Blend voisin de l'Eau), il en reste **1** : cet indice est **redondant**, les 14 autres suffisent à déterminer la solution. Le nouvel exercice 2 pousse cette question un cran plus loin.

### Exercice 2 -- Un sous-ensemble minimal d'indices

**Objectif** : sans l'indice 15, l'énigme garde exactement 1 solution — mais est-il le seul qu'on puisse retirer ? Trouver par **élimination gloutonne** un sous-ensemble des 15 indices qui garde **exactement 1 solution** : partir de la liste complète, et pour chaque indice dans l'ordre, essayer de le retirer **définitivement** — si le reste a toujours exactement 1 solution, il reste retiré, sinon il est remis. Combien d'indices le résultat conserve-t-il, et lesquels ?

**Indices** : listez d'abord les 15 indices sous forme de fermetures `|v: &HashMap<String, Int>| -> Bool` (les deux du bonus donnent le patron) ; reconstruisez à chaque essai un solveur frais ne contenant que les indices retenus, et comptez avec `compter_solutions(&mut s, &v)`. Affichez le nombre d'indices conservés et leur liste. Attention : un indice isolément redondant peut devenir nécessaire une fois d'autres indices retirés — d'où le test à chaque étape.

In [10]:
// EXERCICE 2 : un sous-ensemble minimal d'indices.
// TODO etudiant :
// 1. lister les 15 indices sous forme de fermetures
//    |v: &HashMap<String, Int>| -> Bool
// 2. boucle d'elimination gloutonne : pour chaque indice dans l'ordre,
//    essayer de le retirer definitivement ; si le reste garde exactement
//    1 solution, il reste retire, sinon il est remis
// 3. afficher le nombre d'indices conserves et leur liste
//    (reconstruire a chaque essai un solveur frais, compter avec
//    compter_solutions(&mut s, &v))
println!("Exercice a completer : élimination gloutonne des indices redondants.");


Exercice a completer : élimination gloutonne des indices redondants.


### Exemple guidé 3 -- Mini-Zebra à 4 maisons

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : modéliser une **mini-énigme** à 4 maisons et 3 attributs (4 nationalités, 4 couleurs, 4 animaux) avec 8 indices, et vérifier que l'encodage par position se généralise — le modèle est paramètre par les données, seule l'instance change.

In [11]:
// Exemple guidé 3 : un mini-Zebra a 4 maisons / 3 attributs.
// Indice : groupes_4 = {Nationalite, Couleur, Animal}, domaine 0..3.
// Etape 1 : definir groupes_4 et un solveur avec domaine 0..3 + 3 Distinct.
// Etape 2 : ajouter 8 indices (identites + adjacences).
// Etape 3 : check() -> solution unique ?
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::{SatResult, Solver};

fn resoudre_mini_zebra() -> (Solver, HashMap<String, Int>) {
    // 1. Definition des groupes
    let groupes_4: Vec<(&str, Vec<&str>)> = vec![
        ("Nationalite", vec!["Francais", "Anglais", "Espagnol", "Italien"]),
        ("Couleur", vec!["Rouge", "Bleu", "Vert", "Jaune"]),
        ("Animal", vec!["Chien", "Chat", "Poisson", "Oiseau"]),
    ];

    let mut v: HashMap<String, Int> = HashMap::new();
    for (_groupe, valeurs) in groupes_4.iter() {
        for nom in valeurs {
            v.insert(nom.to_string(), Int::new_const(*nom));
        }
    }

    let mut s = Solver::new();

    // 2. Domaine : chaque variable doit etre entre 0 et 3
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(3)));
    }

    // Les 4 valeurs de chaque attribut doivent etre dans des maisons differentes
    for (_groupe, valeurs) in groupes_4.iter() {
        let vars: Vec<Int> = valeurs.iter().map(|nom| v[*nom].clone()).collect();
        s.assert(&Ast::distinct(&vars));
    }

    // 3. Les 8 indices
    s.assert(&Ast::eq(&v["Francais"], &v["Rouge"]));          // 1. le Francais habite dans la maison rouge
    s.assert(&Ast::eq(&v["Anglais"], &v["Chien"]));           // 2. l'Anglais possede le chien
    s.assert(&Ast::eq(&v["Espagnol"], &v["Vert"]));          // 3. l'Espagnol habite dans la maison verte
    s.assert(&Ast::eq(&v["Francais"], &v["Chat"]));           // 4. le Francais possede le chat
    s.assert(&Ast::eq(&v["Espagnol"], &Int::from_i64(0)));    // 5. l'Espagnol habite dans la premiere maison
    s.assert(&Bool::or(&[                                    // 6. le chien est a cote de la maison jaune
        &Ast::eq(&Int::sub(&[&v["Chien"], &v["Jaune"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Jaune"], &v["Chien"]]), &Int::from_i64(1)),
    ]));
    s.assert(&Ast::eq(&v["Poisson"], &v["Vert"]));           // 7. le poisson est dans la maison verte
    s.assert(&Bool::or(&[                                    // 8. la maison bleue est a cote de la maison verte
        &Ast::eq(&Int::sub(&[&v["Bleu"], &v["Vert"]]), &Int::from_i64(1)),
        &Ast::eq(&Int::sub(&[&v["Vert"], &v["Bleu"]]), &Int::from_i64(1)),
    ]));

    (s, v)
}

// Test du solveur
let (s, v) = resoudre_mini_zebra();

if s.check() == SatResult::Sat {
    match s.get_model() {
        Some(m) => {
            println!("Solution trouvée !");

            // Affichage des maisons
            for maison in 0..4i64 {
                println!("\nMaison {} :", maison + 1);

                for (groupe, valeurs) in [
                    ("Nationalité", vec!["Francais", "Anglais", "Espagnol", "Italien"]),
                    ("Couleur", vec!["Rouge", "Bleu", "Vert", "Jaune"]),
                    ("Animal", vec!["Chien", "Chat", "Poisson", "Oiseau"]),
                ] {
                    for valeur in valeurs {
                        if m.eval(&v[valeur], true).and_then(|x| x.as_i64()) == Some(maison) {
                            println!("  {} : {}", groupe, valeur);
                        }
                    }
                }
            }
        }
        None => println!("Aucune solution."),
    }
} else {
    println!("Aucune solution.");
}

// Etape 3 : solution unique ? On reutilise compter_solutions (exemple guide 2).
let (mut s_u, v_u) = resoudre_mini_zebra();
println!("\nNombre de solutions : {} (attendu : 1)", compter_solutions(&mut s_u, &v_u, 1000));


Solution trouvée !


Maison 1 :


  Nationalité : Espagnol


  Couleur : Vert


  Animal : Poisson


Maison 2 :


... (8 outputs left out)

  Nationalité : Francais


  Couleur : Rouge


  Animal : Chat


Nombre de solutions : 1 (attendu : 1)


### Lecture du résultat : l'encodage se généralise tel quel

Le même encodage par position passe directement à 4 maisons : domaine `0..3`, 3 groupes `Distinct` et 8 indices (égalités de position et adjacences). Trouver une solution (`sat`) ne suffit pas pour une énigme : il faut aussi qu'elle soit **unique**. En réutilisant `compter_solutions` (exemple guidé 2), on vérifie que les 8 indices déterminent **une seule solution** : l'Espagnol, dans la maison verte, possède le poisson.

### Exercice 3 -- Cinq amies, cinq boissons, cinq loisirs

**Objectif** : modéliser une énigme à **5 maisons** sur un autre thème, avec deux types de relations **nouveaux** par rapport au mini-Zebra : « à gauche quelque part » (`a < b`) et « immédiatement à gauche de » (`a == b - 1`). Cinq amies (Alice, Bruno, Chloe, David, Emilie), cinq boissons (Cafe, The, Jus, Lait, Eau) et cinq loisirs (Echecs, Lecture, Course, Peinture, Musique) — une valeur par maison, une maison par valeur.

Les 11 indices :

1. Alice habite la première maison.
2. Emilie habite la dernière maison.
3. Qui boit du lait fait de la peinture.
4. Bruno boit du thé.
5. Le joueur d'échecs habite quelque part à gauche du buveur de café.
6. David boit du jus.
7. Le coureur habite immédiatement à gauche de Chloe.
8. La lectrice boit de l'eau.
9. Chloe fait de la musique.
10. Le buveur de café habite immédiatement à droite de Bruno.
11. La peinture se pratique dans une maison à gauche de celle de Chloe.

**Indices** : reprenez la structure de `resoudre_mini_zebra` (domaine `0..4`, 3 groupes `Distinct`) ; « immédiatement à gauche de » se code `Ast::eq(&x, &Int::sub(&[&y, &Int::from_i64(1)]))` et « à gauche quelque part » `x.lt(&y)`. Vérifiez `sat` **puis** l'unicité avec `compter_solutions(&mut s, &v)` — attendu : exactement 1 solution.

In [12]:
// EXERCICE 3 : cinq amies, cinq boissons, cinq loisirs (5 maisons, 3 attributs, 11 indices).
// TODO etudiant :
// 1. definir groupes_5 (Prenom, Boisson, Loisir), domaine 0..4 + 3 Distinct
// 2. encoder les 11 indices de l'enonce (egalites, adjacences, "a gauche",
//    "immediatement a gauche") :
//    - immediatement a gauche : Ast::eq(&x, &Int::sub(&[&y, &Int::from_i64(1)]))
//    - a gauche quelque part   : x.lt(&y)
// 3. check() -> sat, puis compter_solutions(&mut s, &v) -> attendu exactement 1
println!("Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).");


Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).


## Conclusion

L'enigme d'Einstein illustre le gain du **declaratif** sur une classe de problemes ou l'approche humaine (deduction progressive sur un tableau) et l'approche naive (brute force) sont toutes deux laborieuses :

- L'**encodage par position** (une variable entiere = position de chaque valeur) transforme un puzzle apparemment qualitatif en un système de contraintes arithmetiques : egalites (`Brit == Red`), adjacences (`|Blend - Cat| == 1`), permutations (`Distinct`).
- La **propagation de contraintes** resout les 25 variables entrelacees en millisecondes la ou la brute force affronte ~25 milliards de combinaisons ; le solveur **prouve l'unicite** en niant le temoin et en observant `unsat`.
- Le modèle est **paramètre par les données** : ajouter un indice, un attribut ou une maison se fait en modifiant l'instance, sans re-ecrire l'algorithme de resolution.

**Contraste avec le job-shop (notebook 08)** : ici `Solver` suffit (on cherche une solution realisable, pas un optimum) - c'est la **satisfiabilite** pure, la ou l'ordonnancement relevait de l'**optimisation** (`Optimize.minimize` du makespan). Les deux postures du solveur, posees au notebook 01, trouvent ici leur illustration complementaire.

Suite logique : retour a la [serie Z3-API](README.md), ou comparez avec le jumeau [Python](Z3-09-Enigme-Einstein-Python.ipynb) et la version C# [Z3.Linq](../Z3-Linq2Z3/README.md) qui aborde la même enigme via la couche declarative LINQ.